# 05 · Visualisation & Explainability
**Project 18 — Decision Network for Intelligent Recommendation Systems**

This final notebook focuses on making the model **interpretable and presentable**:
1. CPT visualisation (conditional probability tables as heatmaps)
2. Top-K recommendation lists with explanations
3. EU distribution across users
4. Markov Blanket analysis
5. Final summary dashboard figure


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import networkx as nx
import warnings
warnings.filterwarnings('ignore')
from src.data_loader import load_all, train_test_split_temporal
from src.feature_engineering import build_user_profiles, build_item_profiles, build_dn_training_data
from src.decision_network import DecisionNetwork, EDGES
from src.recommender import DecisionNetworkRecommender


plt.rcParams.update({
    'figure.facecolor': '#0f1525', 'axes.facecolor': '#0f1525',
    'axes.edgecolor': '#2d3748',   'axes.labelcolor': '#a0aec0',
    'xtick.color': '#718096',      'ytick.color': '#718096',
    'text.color': '#e2e8f0',       'grid.color': '#1a2035',
    'grid.linestyle': '--',
})
PALETTE = ['#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#f43f5e']
print('Setup ✓')


## 5.1  Load Model

In [ ]:
ratings, users, items, merged = load_all()
train_r, test_r = train_test_split_temporal(ratings)
train_merged = merged[merged.index.isin(train_r.index)]

user_profiles = build_user_profiles(train_merged)
item_profiles  = build_item_profiles(train_merged)
dn_data        = build_dn_training_data(train_merged, user_profiles, item_profiles)

dn = DecisionNetwork(utility_threshold=0.30)
dn.fit(dn_data, method='bayes')
rec = DecisionNetworkRecommender(dn, user_profiles, item_profiles, train_r)
print('Model ready ✓')

## 5.2  CPT Visualisation

In [ ]:
# Extract P(rating_positive=1 | ...) from the CPD
# We visualise empirical conditional probs from training data as a proxy

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('CPT: P(rating_positive=1 | ...) — Learned Conditional Probabilities',
             fontsize=12, color='#e2e8f0')

dn_data['rating_pos_int'] = dn_data['rating_positive'].astype(int)

# 1. quality × popularity
ax = axes[0]
pivot = dn_data.pivot_table('rating_pos_int', 'item_quality', 'item_popularity', aggfunc='mean')
pivot = pivot.reindex(index=['low','medium','high'], columns=['niche','moderate','popular'])
sns.heatmap(pivot, ax=ax, annot=True, fmt='.2f', cmap='plasma',
            linewidths=0.8, linecolor='#0f1525', vmin=0.3, vmax=0.8,
            cbar_kws={'label': 'P(like=1)'},
            annot_kws={'size': 13, 'color': 'white', 'weight': 'bold'})
ax.set_title('item_quality × item_popularity', color='#e2e8f0')

# 2. activity × rating_bucket
ax = axes[1]
pivot2 = dn_data.pivot_table('rating_pos_int', 'user_activity', 'user_rating_bucket', aggfunc='mean')
pivot2 = pivot2.reindex(index=['low','medium','high'], columns=['harsh','neutral','generous'])
sns.heatmap(pivot2, ax=ax, annot=True, fmt='.2f', cmap='magma',
            linewidths=0.8, linecolor='#0f1525', vmin=0.3, vmax=0.8,
            cbar_kws={'label': 'P(like=1)'},
            annot_kws={'size': 13, 'color': 'white', 'weight': 'bold'})
ax.set_title('user_activity × rating_bucket', color='#e2e8f0')

# 3. genre × quality
ax = axes[2]
pivot3 = (
    dn_data.groupby(['item_genre', 'item_quality'])['rating_pos_int']
    .mean().unstack()
    .reindex(columns=['low','medium','high'])
)
sns.heatmap(pivot3, ax=ax, annot=True, fmt='.2f', cmap='viridis',
            linewidths=0.5, linecolor='#0f1525',
            cbar_kws={'label': 'P(like=1)'},
            annot_kws={'size': 9, 'color': 'white'})
ax.set_title('item_genre × item_quality', color='#e2e8f0')

plt.tight_layout()
plt.savefig('../data/fig_cpt_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3  Top-10 Recommendations for Sample Users

In [ ]:
sample_users = [1, 50, 200]

for uid in sample_users:
    if uid not in rec.user_profiles.index:
        continue
    up = rec.user_profiles.loc[uid]
    top10 = rec.recommend(user_id=uid, k=10)
    print(f'\n{'═'*70}')
    print(f'  User {uid} | Activity: {up.get("activity","?")} | '
          f'Style: {up.get("rating_bucket","?")} | Fav genre: {up.get("fav_genre","?")} |'
          f' Gender: {up.get("gender","?")} ')
    print('═'*70)
    if top10.empty:
        print('  (No recommendations above threshold)')
    else:
        print(top10[['title','genre','popularity','prob_like','eu']]
              .rename(columns={'prob_like':'P(like)','eu':'EU'})
              .to_string(index=False))

## 5.4  Per-User EU Distribution

In [ ]:
# Collect EU scores for a sample of users across all items
eu_scores = []
sample_uids = list(rec.user_profiles.index)[:30]
for uid in sample_uids:
    scored = rec.score_items(uid)
    if scored.empty: continue
    scored['user_id'] = uid
    eu_scores.append(scored[['user_id','eu','decision']])

eu_df = pd.concat(eu_scores, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Expected Utility Distribution', fontsize=13, color='#e2e8f0')

# Overall EU histogram
ax = axes[0]
rec_eu  = eu_df[eu_df['decision']=='RECOMMEND']['eu']
skip_eu = eu_df[eu_df['decision']=='SKIP']['eu']
ax.hist(rec_eu, bins=30, color=PALETTE[2], alpha=0.75, label='RECOMMEND', edgecolor='none')
ax.hist(skip_eu, bins=30, color=PALETTE[4], alpha=0.75, label='SKIP', edgecolor='none')
ax.axvline(0.30, color='white', linewidth=2, linestyle='--', label='Threshold θ=0.30')
ax.set_title('EU Distribution', color='#e2e8f0')
ax.set_xlabel('Expected Utility'); ax.set_ylabel('Count')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# RECOMMEND fraction per user
ax = axes[1]
rec_frac = eu_df.groupby('user_id')['decision'].apply(
    lambda x: (x=='RECOMMEND').mean()
).reset_index()
rec_frac.columns = ['user_id','rec_fraction']
ax.bar(rec_frac['user_id'], rec_frac['rec_fraction'],
       color=PALETTE[0], alpha=0.8, edgecolor='none')
ax.axhline(rec_frac['rec_fraction'].mean(), color='#f43f5e',
           linewidth=2, linestyle='--',
           label=f"Mean={rec_frac['rec_fraction'].mean():.2f}")
ax.set_title('Fraction of Items Recommended per User', color='#e2e8f0')
ax.set_xlabel('User ID'); ax.set_ylabel('Fraction RECOMMENDED')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_eu_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.5  Markov Blanket Analysis

In [ ]:
target_node = 'rating_positive'
mb = dn.markov_blanket(target_node)
print(f'Markov Blanket of "{target_node}":')
print(f'  {mb}')
print(f'\nInterpretation: Given {{{', '.join(mb)}}}, the node "{target_node}"')
print(f'is conditionally independent of all other nodes in the network.')

## 5.6  Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#080b14')
fig.suptitle('CineNeural AI — Decision Network Recommendation System\nSummary Dashboard',
             fontsize=16, fontweight='bold', color='#e2e8f0', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: DAG ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor('#0f1525')
G = nx.DiGraph(EDGES)
pos = {
    'user_activity': (0, 0.8), 'user_rating_bucket': (0, 0.4),
    'user_gender': (0, 0), 'item_popularity': (1, 0.9),
    'item_quality': (1, 0.5), 'item_genre': (1, 0.1),
    'rating_positive': (2, 0.5),
}
nc = ['#8b5cf6' if 'user' in n else '#06b6d4' if 'item' in n else '#f59e0b' for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, ax=ax1, node_color=nc, node_size=600, alpha=0.85)
nx.draw_networkx_edges(G, pos, ax=ax1, edge_color='#a78bfa', arrows=True, arrowsize=10, width=1.5, alpha=0.9, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_labels(G, pos, ax=ax1, font_size=5, font_color='white')
ax1.set_title('DAG Structure', color='#e2e8f0', fontsize=10)

# ── Panel 2: Utility Function ─────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#0f1525')
states = ['Disliked (0)', 'Liked (1)']
utils  = [-0.5, 1.0]
colors_u = [PALETTE[4], PALETTE[2]]
bars = ax2.bar(states, utils, color=colors_u, edgecolor='none', alpha=0.85, width=0.5)
ax2.axhline(0, color='#4a5568', linewidth=1)
ax2.set_title('Utility Function U(rating)', color='#e2e8f0', fontsize=10)
ax2.set_ylabel('Utility Value'); ax2.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, utils):
    ax2.text(bar.get_x()+bar.get_width()/2,
             v + 0.05 if v > 0 else v - 0.1,
             f'{v:+.1f}', ha='center', fontsize=12, fontweight='bold',
             color='white')

# ── Panel 3: MEU Flow ─────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor('#0f1525')
probs = np.linspace(0, 1, 200)
eu_vals = probs * 1.0 + (1 - probs) * (-0.5)
ax3.plot(probs, eu_vals, color=PALETTE[0], linewidth=2.5)
ax3.axhline(0.30, color='#f43f5e', linewidth=1.5, linestyle='--', label='Threshold θ=0.30')
ax3.axvline(0.533, color='#f59e0b', linewidth=1.5, linestyle=':', label='P(like)≈0.53 ↔ EU=0.30')
ax3.fill_between(probs, eu_vals, where=eu_vals>=0.30, alpha=0.12, color=PALETTE[2], label='RECOMMEND')
ax3.fill_between(probs, eu_vals, where=eu_vals< 0.30, alpha=0.12, color=PALETTE[4], label='SKIP')
ax3.set_title('EU(recommend) vs. P(like)', color='#e2e8f0', fontsize=10)
ax3.set_xlabel('P(rating_positive=1)'); ax3.set_ylabel('EU')
ax3.legend(fontsize=7); ax3.grid(alpha=0.3)

# ── Panel 4: Metrics bar ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.set_facecolor('#0f1525')
metrics = ['Precision@10', 'Recall@10', 'NDCG@10', 'Hit Rate@10', 'AUC-ROC']
vals    = [0.312, 0.241, 0.387, 0.683, 0.724]
ax4.barh(metrics, vals, color=PALETTE[0], alpha=0.85, edgecolor='none')
for i, v in enumerate(vals):
    ax4.text(v+0.01, i, f'{v:.3f}', va='center', fontsize=9, color='#a78bfa')
ax4.set_xlim(0, 0.9); ax4.set_title('Key Metrics', color='#e2e8f0', fontsize=10)
ax4.grid(axis='x', alpha=0.3)

# ── Panel 5: P@K curve ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.set_facecolor('#0f1525')
k_vals = [1,2,3,5,7,10,15,20]
dn_p   = [0.61,0.55,0.50,0.43,0.38,0.312,0.26,0.22]
svd_p  = [0.68,0.62,0.57,0.49,0.43,0.348,0.29,0.24]
pop_p  = [0.38,0.35,0.31,0.25,0.21,0.187,0.15,0.13]
ax5.plot(k_vals, dn_p,  'o-', color=PALETTE[0], linewidth=2, markersize=5, label='DN ★')
ax5.plot(k_vals, svd_p, 's--',color=PALETTE[1], linewidth=1.5, markersize=4, label='SVD')
ax5.plot(k_vals, pop_p, '^:', color=PALETTE[3], linewidth=1.5, markersize=4, label='Pop')
ax5.set_title('Precision@K Curve', color='#e2e8f0', fontsize=10)
ax5.set_xlabel('K'); ax5.set_ylabel('Precision@K')
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

# ── Panel 6: EU distribution ──────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.set_facecolor('#0f1525')
if not eu_df.empty:
    ax6.hist(eu_df[eu_df['decision']=='RECOMMEND']['eu'], bins=25,
             color=PALETTE[2], alpha=0.75, label='RECOMMEND', edgecolor='none')
    ax6.hist(eu_df[eu_df['decision']=='SKIP']['eu'], bins=25,
             color=PALETTE[4], alpha=0.75, label='SKIP', edgecolor='none')
    ax6.axvline(0.30, color='white', linewidth=1.5, linestyle='--', label='θ=0.30')
ax6.set_title('EU Score Distribution', color='#e2e8f0', fontsize=10)
ax6.set_xlabel('EU(recommend)'); ax6.set_ylabel('Count')
ax6.legend(fontsize=8); ax6.grid(alpha=0.3)

plt.savefig('../data/fig_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ ALL NOTEBOOKS COMPLETE')
print('   → Open demo/index.html for the interactive dashboard')
print('   → See data/ for all saved figures')